# Практика · Question answering

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє: [homework.html](homework.html)

> ⏱ Зошит будує набір питань із твоєї системи й навчає **сто сімдесят** лінійних
> моделей: сорок розбиттів × дві моделі на великому наборі, стільки ж на
> вужчому, плюс десять окремих прогонів.
> Заміряно: **131 секунда процесорного часу** на чотирьох ядрах без відеокарти,
> **в один потік**. Стінного часу буде помітно більше: на нашій машині
> паралельно рахувалося кілька чужих завдань, середнє навантаження сягало 14, і
> прогін брав понад пів години. Останньою клітинкою зошит друкує власний
> процесорний час — звіряй саме з ним, а не з годинником.

## Що ми тут робимо

**Question answering** — це коли машині дають текст і питання про нього, а вона
має відповісти. Найпоширеніший різновид — **екстрактивний**: відповідь не
пишеться, а **знаходиться в тексті** як суцільний шматок (проміжок, span).

Щоб таку систему навчити чи перевірити, потрібен набір троек «уривок → питання →
відповідь». Ми зробимо його **самі, з описів пакетів, які стоять у тебе в
системі**, і на ньому послідовно відповімо на три питання:

1. **Скільки в такому тесті можна набрати, взагалі не читаючи питання?**
   Це головна перевірка будь-якого набору для QA, і вона майже завжди дає
   неприємне число.
2. **Чи вміє щось більше система, яка питання таки читає?** Побудуємо
   екстрактивний добирач на ознаках і поставимо його поруч із тупими рубежами.
3. **Чи справді вона читає?** Підсунемо їй **чуже** питання й подивимось, чи
   впаде точність. Якщо не впаде — вона питання не читала, хай би що показувала
   таблиця.

Наприкінці додамо **питання без відповіді** й побачимо, що система, яку не вчили
мовчати, вигадує відповідь завжди.

⚠️ **Числа в тебе будуть інші, ніж у нас.** Набір пакетів у кожній машині свій.
Порядок величин і **напрямок** усіх висновків відтворюються, конкретні цифри — ні.
Тому зошит друкує **свої** числа, а не звіряється з лекційними.

## 0 · Середовище: чому спершу фіксуємо потоки

Зошит міряє час, і час на спільній машині бреше двома способами.

**Стінний годинник** (`time.time()`) показує, скільки минуло реального часу.
Якщо поруч рахується щось інше, він покаже більше — і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує лише той час, коли
процесор працював саме над нашою програмою. Але й у нього є пастка: якщо
бібліотека лінійної алгебри розкладає роботу на кілька потоків, то потоки, які
**чекають**, теж зараховуються як робота. Тому спершу наказуємо їй працювати в
один потік — і робимо це **до** імпорту `numpy`, бо змінні середовища читаються
в момент завантаження бібліотеки.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту numpy: інакше бібліотеки вже
# запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import re, sys, math, time, random, subprocess, collections, statistics
import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

START = time.process_time()          # звідси рахуємо власний час зошита

print('python              ', sys.version.split()[0])
print('numpy               ', np.__version__)
print('scikit-learn        ', sklearn.__version__)
print('ядер у машині       ', os.cpu_count())
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · Звідки беремо тексти

Курс не тримає даних у репозиторії. Тексти ми беремо **з твоєї системи**: у
кожного встановленого пакета є опис, який писала людина — супровідник пакета.

Дві потрібні нам речі лежать у полях бази пакетів:

- `%{SUMMARY}` — один рядок, короткий опис («A TLS protocol implementation»);
- `%{DESCRIPTION}` — абзац на кілька речень.

Це справжній технічний текст: із назвами програм, форматів, ліцензій і
протоколів. Саме такий текст і потрібен для питань «що це за штука».

⚠️ **Головне попередження зошита.** Ми читаємо базу `rpm` (Fedora, RHEL,
openSUSE). На Debian і Ubuntu та сама інформація живе в `dpkg`, і запасний шлях
нижче її звідти дістає. **Ми цей шлях не перевіряли** — Debian-машини під рукою
не було, і чесніше сказати це прямо, ніж вдавати, що перевірили. Якщо ти на
Debian і щось піде не так, дивись саме туди.

Якщо жодна з двох баз не відповідає, зошит скаже це людською мовою й зупиниться —
без стека помилок на пів екрана.

In [ ]:
FIELDS = ('NAME', 'SUMMARY', 'LICENSE', 'URL', 'DESCRIPTION')
MIN_WORDS = 20               # опис коротший за 20 слів для питань не годиться


def read_rpm():
    """Опитує базу rpm. Розділювачі \x1f і \x1e — це керівні символи,
    яких у людському тексті не буває, тож поля не сплутаються."""
    fmt = '\x1f'.join('%{' + f + '}' for f in FIELDS) + '\x1e'
    out = subprocess.run(['rpm', '-qa', '--qf', fmt],
                         capture_output=True, text=True, timeout=180)
    return out.stdout


def read_dpkg():
    """Той самий запит до бази Debian. ⚠️ ЦЕЙ ШЛЯХ МИ НЕ ПЕРЕВІРЯЛИ:
    Debian-машини в нас не було. У dpkg немає окремого поля LICENSE, тож
    ліцензія лишиться порожньою, а Description містить і короткий рядок,
    і абзац — перший рядок відрізаємо як резюме."""
    fmt = '${Package}\x1f${binary:Summary}\x1f\x1f${Homepage}\x1f${Description}\x1e'
    out = subprocess.run(['dpkg-query', '-W', '-f', fmt],
                         capture_output=True, text=True, timeout=180)
    return out.stdout


def load_packages():
    """Повертає список словників або пояснює людською мовою, чому не вийшло."""
    raw = ''
    for reader, name in ((read_rpm, 'rpm'), (read_dpkg, 'dpkg')):
        try:
            raw = reader()
        except (FileNotFoundError, subprocess.SubprocessError):
            continue
        if raw.strip():
            print(f'база пакетів: {name}')
            break
    if not raw.strip():
        raise SystemExit(
            'Не вдалося прочитати базу встановлених пакетів.\n'
            'Зошит працює там, де є rpm (Fedora, RHEL, openSUSE) або\n'
            'dpkg (Debian, Ubuntu). Перевір у терміналі:\n'
            '    rpm -qa | head\n'
            '    dpkg-query -W | head\n'
            'Якщо в тебе інша система — заміни ці два виклики на будь-яке\n'
            'джерело текстів із полями «назва / короткий опис / абзац опису».')

    packages = []
    for record in raw.split('\x1e'):
        parts = record.split('\x1f')
        if len(parts) != len(FIELDS):
            continue
        pkg = dict(zip((f.lower() for f in FIELDS), (p.strip() for p in parts)))
        pkg['desc'] = pkg.pop('description')
        # відкидаємо описи-заглушки й надто короткі: у них нема про що питати
        if pkg['desc'] and pkg['desc'] != '(none)' and len(pkg['desc'].split()) >= MIN_WORDS:
            packages.append(pkg)
    packages.sort(key=lambda p: p['name'])
    return packages


packages = load_packages()
if len(packages) < 200:
    raise SystemExit(
        f'Знайшлося лише {len(packages)} пакетів з описом на {MIN_WORDS}+ слів.\n'
        'Для навчання й розбиття за пакетом цього замало — потрібно хоча б 200.\n'
        'Так буває в мінімальних контейнерах, де описи вирізано.')

print('пакетів з описом ≥', MIN_WORDS, 'слів:', len(packages))
print('слів в описі: медіана',
      int(statistics.median(len(p['desc'].split()) for p in packages)))

Подивимось на один запис цілком — щоб було видно, з чим працюємо.

In [ ]:
example = next(p for p in packages if len(p['desc'].split()) > 45)
print('назва   :', example['name'])
print('резюме  :', example['summary'])
print('ліцензія:', example['license'])
print('сайт    :', example['url'])
print('опис    :', example['desc'][:400], '…')

## 2 · Речення й сутності

Далі знадобляться дві прості речі.

**Поділ на речення.** Ріжемо по крапці, знаку оклику чи питання, за якими йде
пробіл і велика літера або цифра. Це груба евристика, і на скороченнях («i.e.»,
«e.g.») вона іноді помиляється — але для нашої задачі цього досить, а складніший
поділ вимагав би моделі, якої тут немає.

**Сутності.** Нам потрібні шматки тексту, які можуть бути **відповіддю**:
назви програм, форматів, протоколів, ліцензій. Розпізнаємо їх двома
незалежними способами.

*Спосіб перший — газетир із типізованих полів тієї самої бази.* Ми не вигадуємо
розмітку: імена пакетів (`%{NAME}`) дають тип `PROD`, ідентифікатори ліцензій
(`%{LICENSE}`) — тип `LIC`, домен із `%{URL}` — тип `ORG`. Ці мітки написала
людина, просто в іншому полі. Така розмітка зветься **срібною** (silver): вона
не еталонна, але й не вигадана.

*Спосіб другий — велика літера.* У технічному описі слово з великої літери
посеред речення майже завжди є власною назвою: `TLS`, `PCRE2`, `XeTeX`, `Noto`.
Викидаємо службові слова («The», «This», «However»), решту беремо як сутність
типу `CAP`.

Другий спосіб грубіший — і саме тому корисний: без нього кандидатів у
кожному уривку буде замало, і тест вийде надто легким. Нижче ми це побачимо числом.

In [ ]:
WORD = re.compile(r"[\w][\w.+-]*|[^\s\w]")


def tokens(text):
    return WORD.findall(text)


def sentences(text):
    """Ріжемо на речення. Крапка + пробіл + велика літера або цифра."""
    flat = re.sub(r'\s+', ' ', text).strip()
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', flat)
    return [s.strip() for s in parts if s.strip()]


# службові слова, які часто стоять з великої літери, але власними назвами не є
CAP_STOP = set(
    'The This These Those There It A An In On For To Of And Or But If When While '
    'You Your We Our They Their He She His Her Its As At By From With Without Not '
    'No All Some Any Each Both Also However Note NOTE See Use Used Using Is Are Was '
    'Were Be Been Being Can Could Should Would May Might Must Do Does Did Has Have '
    'Had Will Shall Since Because So Than Then Thus Such Other Another More Most '
    'Many Much Few Several Only Even Just Now Here How What Which Who Whom Whose '
    'Where Why Additionally Moreover Furthermore Currently Please Warning I II III'.split())

CAP_RE = re.compile(r'(?<![\w-])([A-Z][A-Za-z0-9]*(?:[+._-][A-Za-z0-9]+)*)(?![\w-])')


def build_gazetteer(packages, org_mode='fixed'):
    """Три множини рядків із типізованих полів бази.

    org_mode='fixed' — організацію беремо з РЕЄСТРОВАНОЇ частини хоста;
    org_mode='naive' — з першої частини, як здається природним. Наївний
    варіант лишаємо навмисне: нижче ми заміряємо, у скільки він обходиться.
    """
    names = {p['name'] for p in packages}
    # урізаний префікс (open-vm-tools → open) беремо лише тоді, коли він сам
    # є іменем пакета — інакше в газетир потраплять звичайні слова
    products = {n for n in names if len(n) > 3}
    products |= {p['name'].split('-')[0] for p in packages} & names
    # службові англійські слова, що випадково збігаються з іменами пакетів
    junk = {'which', 'make', 'less', 'file', 'time', 'info', 'more', 'base',
            'text', 'data', 'tools', 'library', 'system', 'open', 'source',
            'common', 'util'}
    products = {n for n in products if n.lower() not in junk}

    licenses = set()
    for p in packages:
        for piece in re.split(r'\s+(?:AND|OR|and|or|WITH)\s+|[()]', p['license'] or ''):
            piece = piece.strip()
            if len(piece) > 3:
                licenses.add(piece)

    # у ccTLD другий рівень службовий: sourceforge.co.uk → потрібен третій з кінця
    SERVICE_LEVEL = {'co', 'com', 'org', 'net', 'ac', 'gov', 'edu'}
    orgs = set()
    for p in packages:
        m = re.match(r'https?://(?:www\.)?([^/:]+)', p['url'] or '')
        if not m:
            continue
        parts = [x for x in m.group(1).split('.') if x]
        if len(parts) < 2:
            continue
        if org_mode == 'naive':
            host = parts[0]
        else:
            host = parts[-2]
            if host in SERVICE_LEVEL and len(parts) >= 3:
                host = parts[-3]
        if len(host) > 3:
            orgs.add(host)
    return {'PROD': products, 'LIC': licenses, 'ORG': orgs}


def as_pattern(strings):
    """Одна регулярка на всю множину. Довші рядки — першими, щоб
    'GPL-2.0-or-later' виграв у 'GPL'."""
    body = '|'.join(re.escape(s) for s in sorted(strings, key=len, reverse=True))
    return re.compile(r'(?<![\w-])(' + body + r')(?![\w-])')


def make_patterns(gaz):
    return {kind: as_pattern(values) for kind, values in gaz.items()}


gazetteer = build_gazetteer(packages, 'fixed')
gazetteer_naive = build_gazetteer(packages, 'naive')
patterns = make_patterns(gazetteer)
patterns_naive = make_patterns(gazetteer_naive)

print('газетир виправлений:', {k: len(v) for k, v in gazetteer.items()})
print('газетир наївний    :', {k: len(v) for k, v in gazetteer_naive.items()})
print()
print('що наївний варіант додає до списку організацій (перші 16):')
print(sorted(gazetteer_naive['ORG'] - gazetteer['ORG'])[:16])

Два списки організацій різні, і різниця не косметична.

`%{URL}` пакета — адреса його сайту. Здається природним узяти з
`https://fonts.google.com/noto` **перше** слово хоста й дістати «fonts».
Насправді організація тут `google`, а `fonts` — звичайне англійське слово, яке
після цього позначатиметься як **назва організації** в кожному описі, де
трапиться. Один запис із хибним розбором отруює сотні міток по всьому корпусу.

Правильна частина хоста — **передостання**: `google` у `fonts.google.com`,
`debian` у `packages.qa.debian.org`. Для доменів на кшталт `.co.uk` беремо
третю з кінця, бо друга там службова.

Нижче ми заміряємо, у скільки обходиться ця однорядкова помилка. Це головна
думка теми в мініатюрі: **тест на розуміння настільки хороший, наскільки хороше
те, з чого його зліпили** — і жодна модель цього не виправить.

In [ ]:
# пріоритет типу за однакового проміжку: рядок «Python» стоїть і в полі
# ліцензії, і серед імен пакетів, але в тексті опису це майже завжди продукт
TYPE_PRIORITY = {'PROD': 0, 'ORG': 1, 'LIC': 2, 'CAP': 3}


def find_entities(text, pats=None, use_caps=True):
    """Усі входження сутностей: (початок, кінець, рядок, тип).
    Збіги, що накладаються, розводимо: виграє той, що починається раніше,
    за рівного початку — довший, за рівної довжини — старший за типом."""
    pats = patterns if pats is None else pats
    found = []
    for kind, pattern in pats.items():
        for m in pattern.finditer(text):
            found.append((m.start(), m.end(), m.group(), kind))
    if use_caps:
        for m in CAP_RE.finditer(text):
            word = m.group()
            if word not in CAP_STOP and len(word) >= 3:
                found.append((m.start(), m.end(), word, 'CAP'))

    found.sort(key=lambda s: (s[0], -(s[1] - s[0]), TYPE_PRIORITY.get(s[3], 9)))
    kept = []
    last_end = -1
    for start, end, string, kind in found:
        if start >= last_end:
            kept.append((start, end, string, kind))
            last_end = end
    return kept


print('приклад речення:')
print(' ', example['desc'][:160])
print('знайдені сутності:')
for start, end, string, kind in find_entities(example['desc'][:160]):
    print(f'   {string:<22} {kind}')

## 3 · Як зробити питання без того, щоб їх хтось писав

Розмічених наборів питань українською або для нашої предметної області немає, і
платити людям за розмітку ми не можемо. Тому користуємось прийомом, який 2015
року дав перший великий набір для машинного читання: **питання з пропуском**
(cloze).

Рецепт із трьох кроків:

1. беремо **одне речення** тексту й викидаємо з нього сутність, лишаючи пропуск
   `@blank` — це **питання**;
2. решта тексту стає **уривком** (passage);
3. викинута сутність — **відповідь**, і вона мусить траплятись в уривку,
   інакше питання нерозвʼязне.

Крок 3 — не формальність. Без нього система, яка чесно шукає відповідь у тексті,
не має жодного шансу, і тест міряв би не читання, а вгадування.

Джерел питань у нас **два**, і обидва — справжній людський текст:

- **речення опису**: питання з одного речення, уривок — усі решта;
- **резюме пакета**: питання з рядка `%{SUMMARY}`, уривок — увесь опис. Це
  окремий текст, написаний тією самою людиною іншими словами, і саме тому
  цінний.

In [ ]:
def make_questions(packages, pats=None, use_caps=True, use_summary=True):
    """Список питань. Кожне — словник із уривком, питанням і відповіддю."""
    items = []

    def add(pkg, source, passage, question, answer, kind):
        occurrences = find_entities(passage, pats, use_caps)
        candidates = sorted({o[2] for o in occurrences})
        if answer not in candidates:
            return                       # відповіді немає в уривку — питання нерозвʼязне
        items.append({'pkg': pkg, 'src': source, 'passage': passage,
                      'occ': occurrences, 'q': question,
                      'ans': answer, 'type': kind, 'cands': candidates})

    for p in packages:
        parts = sentences(p['desc'])
        # джерело 1: одне речення опису стає питанням, решта — уривком
        if len(parts) >= 2:
            for i, sentence in enumerate(parts):
                passage = ' '.join(parts[:i] + parts[i + 1:])
                for start, end, string, kind in find_entities(sentence, pats, use_caps):
                    question = sentence[:start] + '@blank' + sentence[end:]
                    add(p['name'], 'опис', passage, question, string, kind)
        # джерело 2: резюме пакета стає питанням, уривок — увесь опис
        summary = p['summary']
        if use_summary and summary and summary != '(none)':
            passage = ' '.join(parts)
            for start, end, string, kind in find_entities(summary, pats, use_caps):
                question = summary[:start] + '@blank' + summary[end:]
                add(p['name'], 'резюме', passage, question, string, kind)
    return items


questions = make_questions(packages)
counts = [len(q['cands']) for q in questions]
free = [q for q in questions if len(q['cands']) == 1]
real = [q for q in questions if len(q['cands']) > 1]

print('питань усього            ', len(questions))
print('  з них із опису         ', sum(1 for q in questions if q['src'] == 'опис'))
print('  з них із резюме        ', sum(1 for q in questions if q['src'] == 'резюме'))
print('кандидатів в уривку      : медіана', int(statistics.median(counts)),
      '· середнє', round(statistics.mean(counts), 2), '· максимум', max(counts))
print()
print('питань з ЄДИНИМ кандидатом (відповідь безкоштовна):',
      len(free), '=', f'{len(free)/len(questions):.2%}')
print('лишається змістовних питань:', len(real))
print('пакетів серед змістовних   :', len({q['pkg'] for q in real}))

Питання з **єдиним** кандидатом — це пастка, через яку майже всі звіти про QA
виглядають краще, ніж є. Якщо в уривку є рівно одна сутність, то будь-яка
система, що обирає з кандидатів, відповість правильно, нічого не зрозумівши.
Такі питання завищують **усі** числа однаково й тому нічого не розрізняють.

Далі ми міряємо **тільки на змістовних** питаннях — тих, де кандидатів
щонайменше два. Це не косметика: нижче видно, наскільки два стовпчики
розходяться.

Подивимось на три випадкові змістовні питання.

In [ ]:
for q in random.Random(7).sample(real, 3):
    print('пакет      :', q['pkg'], '· джерело:', q['src'], '· тип відповіді:', q['type'])
    print('питання    :', q['q'][:180])
    print('уривок     :', q['passage'][:200], '…')
    print('кандидати  :', q['cands'][:10])
    print('відповідь  :', q['ans'])
    print()

## 4 · Скільки можна набрати, не читаючи питання

Це перше, що треба зробити з будь-яким набором для QA, — **до** того, як щось
навчати. Побудуємо чотири способи відповідати, жоден з яких на питання навіть
не дивиться:

- **випадковий кандидат** — тицяємо навмання;
- **найдовший кандидат** — беремо найдовший рядок;
- **перший кандидат** — беремо сутність, що трапилась в уривку раніше за всіх;
- **найчастіший кандидат** — беремо ту, що трапляється в уривку найчастіше.

Для випадкового вибору **не** кидаємо кубик, а рахуємо **точне очікування**:
якщо в питанні k кандидатів, шанс угадати — 1/k, і середнє цих дробів і є
очікуваною точністю. Один кидок дав би число, яке щоразу інше, і на нього не
можна послатись.

In [ ]:
def pick_random_expected(items):
    """Очікувана точність випадкового вибору: середнє від 1/кількість кандидатів."""
    return statistics.mean(1 / len(q['cands']) for q in items)


def pick_longest(q):
    return max(q['cands'], key=len)


def pick_first(q):
    return q['occ'][0][2]


def pick_most_frequent(q):
    counter = collections.Counter(o[2] for o in q['occ'])
    return counter.most_common(1)[0][0]


def accuracy(picker, items):
    return sum(picker(q) == q['ans'] for q in items) / len(items)


print(f'{"спосіб (питання не читає)":28} {"усі питання":>12} {"тільки змістовні":>18}')
print(f'{"випадковий кандидат":28} {pick_random_expected(questions):12.4f} '
      f'{pick_random_expected(real):18.4f}')
for name, picker in (('найдовший кандидат', pick_longest),
                     ('перший кандидат у тексті', pick_first),
                     ('найчастіший кандидат', pick_most_frequent)):
    print(f'{name:28} {accuracy(picker, questions):12.4f} {accuracy(picker, real):18.4f}')

### Скільки коштує одна помилка в метаданих

Тепер зберемо той самий набір ще тричі, міняючи **лише розпізнавач сутностей**,
і подивимось, як від цього рухаються всі числа одразу.

- **наївний газетир, без великих літер** — те, з чого починають: імена пакетів,
  ліцензії та «організації», узяті з першого слова хоста;
- **виправлений газетир, без великих літер** — та сама конструкція, але
  організація береться з реєстрованої частини хоста;
- **виправлений газетир + великі літери** — наш робочий набір.

Дивитись треба на три колонки: скільки вийшло питань, яка частка **безкоштовних**
(з єдиним кандидатом) і що набирає найтупіший розумний рубіж — «найчастіший
кандидат». Останнє число і є **складністю набору**: що воно вище, то менше
в тесті лишилось для розуміння.

In [ ]:
def describe(label, items):
    real_items = [q for q in items if len(q['cands']) > 1]
    counts_ = [len(q['cands']) for q in items]
    print(f'{label:34} {len(items):>7} '
          f'{(len(items)-len(real_items))/len(items):>13.2%} '
          f'{len(real_items):>11} {statistics.mean(counts_):>11.2f} '
          f'{accuracy(pick_most_frequent, real_items):>12.4f}')


naive_plain = make_questions(packages, patterns_naive, use_caps=False, use_summary=False)
fixed_plain = make_questions(packages, patterns, use_caps=False, use_summary=False)

print(f'{"розпізнавач сутностей":34} {"питань":>7} {"безкоштовних":>14} '
      f'{"змістовних":>11} {"кандидатів":>11} {"найчастіший":>12}')
describe('наївний газетир', naive_plain)
describe('виправлений газетир', fixed_plain)
describe('виправлений + великі літери', questions)

Прочитай цю таблицю уважно, бо в ній три різні уроки.

**Перший.** Виправлення одного рядка в розборі адреси змінює **всі** числа
набору. Це не «шум у даних» — це зсув самого тесту. Якби ми надрукували
точність моделі до й після, різниця виглядала б як покращення чи погіршення
моделі, хоча модель не змінилась ані на біт.

**Другий.** Виправлення робить набір **меншим і легшим**: кандидатів у уривку
поменшало, отже вгадати простіше. Правильніша розмітка дала **гірший тест** —
і це нормально, бо мета розмітки не в тому, щоб тест був складний.

**Третій.** Слова з великої літери повертають складність назад, і саме тому
робочий набір ми будуємо з ними. Але платимо за це грубішою розміткою: серед
«сутностей» будуть слова на початку речення, що власними назвами не є.

Далі всюди працюємо з третім рядком.

Різниця між двома стовпчиками — це ціна безкоштовних питань. Права колонка й
є справжньою складністю набору; ліва — те, що потрапило б у звіт, якби ми
не подивились на кількість кандидатів.

Подивимось на це докладніше: точність кожного способу окремо для питань із
двома, трьома, чотирма й більше кандидатами.

In [ ]:
def group_by_candidates(items):
    groups = collections.defaultdict(list)
    for q in items:
        key = min(len(q['cands']), 6)      # усе, що 6 і більше, в одну купу
        groups[key].append(q)
    return groups


groups = group_by_candidates(questions)
print(f'{"кандидатів":>10} {"питань":>7} {"випадковий":>11} {"найчастіший":>12}')
for key in sorted(groups):
    bucket = groups[key]
    label = f'{key}+' if key == 6 else str(key)
    print(f'{label:>10} {len(bucket):>7} {pick_random_expected(bucket):11.4f} '
          f'{accuracy(pick_most_frequent, bucket):12.4f}')

## 5 · Тепер — система, яка питання таки читає

Тупі рубежі виміряно. Тепер побудуємо **екстрактивний добирач**: він оцінює
кожного кандидата числом і бере найбільше.

Ознаки поділимо на дві групи, і цей поділ — головна конструкція зошита.

**Група PASS** — ознаки, що описують кандидата в уривку й **питання не бачать
узагалі**: як часто трапляється, де стоїть, який завдовжки, якого типу,
наскільки поширений у корпусі.

**Група QDEP** — ознаки, що порівнюють кандидата **з питанням**: скільки слів
питання стоїть поруч із ним, чи збігаються сусіди зліва й справа від пропуску
із сусідами кандидата, чи сам кандидат уже згаданий у питанні, як далеко він
від найближчого слова питання.

Такий поділ дає нам дві безкоштовні речі. По-перше, можна навчити модель
**лише на PASS** — це чесна межа «скільки дає нечитання». По-друге, підмінивши
питання, ми зіпсуємо рівно ознаки QDEP і нічого більше, тож контроль вимірює
саме читання.

In [ ]:
STOP_WORDS = set(
    'a an the of and or to in on for with is are was were be been it its this that '
    'these those as at by from not no can could will would may might must do does '
    'did has have had also which who what when where how than then such other more '
    'most any all both each some'.split())

PASS_FEATURES = ['log_freq', 'pos_first', 'pos_last', 'is_first', 'log_len',
                 'is_upper', 'n_words', 'log_df', 't_PROD', 't_LIC', 't_ORG', 't_CAP']
QDEP_FEATURES = ['ctx_overlap', 'left_match', 'right_match', 'in_question',
                 'near_qword', 'sent_overlap', 'shared_prefix']
ALL_FEATURES = PASS_FEATURES + QDEP_FEATURES
INDEX_OF = {name: i for i, name in enumerate(ALL_FEATURES)}


def index_passage(item):
    """Один раз на питання: слова уривка й позиції кожного кандидата в них."""
    spans = [(m.group(), m.start()) for m in WORD.finditer(item['passage'])]
    words = [w.lower() for w, _ in spans]
    starts = [s for _, s in spans]
    where = collections.defaultdict(list)
    position_of = {s: i for i, s in enumerate(starts)}
    for start, end, string, kind in item['occ']:
        i = position_of.get(start)
        if i is not None:
            where[string].append((i, kind))
    return words, where


def split_question(text):
    """Слова питання, три слова зліва й справа від пропуску, змістовні слова."""
    parts = [w.lower() for w in tokens(text)]
    i = parts.index('@blank') if '@blank' in parts else len(parts) // 2
    left = parts[max(0, i - 3):i]
    right = parts[i + 1:i + 4]
    content = {w for w in parts
               if w != '@blank' and w not in STOP_WORDS and len(w) > 1}
    return set(parts), left, right, content


def document_frequency(items):
    """У скількох різних уривках трапляється кандидат. Від питання не залежить."""
    counter = collections.Counter()
    seen = set()
    for q in items:
        if q['pkg'] in seen:
            continue
        seen.add(q['pkg'])
        for c in {w.lower() for w in q['cands']}:
            counter[c] += 1
    return counter, len(seen)


DF, N_DOCS = document_frequency(questions)
print('різних кандидатів у корпусі:', len(DF))

In [ ]:
def features(item, question_text=None):
    """Рядок ознак на кожного кандидата. Повертає (кандидати, матриця).

    question_text дає змогу підсунути ЧУЖЕ питання, не чіпаючи уривка, —
    саме цим ми потім перевіримо, чи модель питання взагалі читає."""
    words, where = index_passage(item)
    n_words = max(len(words), 1)
    text = item['q'] if question_text is None else question_text
    q_all, q_left, q_right, q_content = split_question(text)
    # позиції слів уривка, які є і в питанні, — для ознаки «стоїть поруч»
    q_positions = [i for i, w in enumerate(words) if w in q_content]

    rows = []
    for candidate in item['cands']:
        places = where.get(candidate, [(0, 'CAP')])
        kind = places[0][1]
        at = [i for i, _ in places]
        low = candidate.lower()

        # --- PASS: питання не бачимо ---
        row = [
            math.log1p(len(at)),                       # як часто трапляється
            at[0] / n_words,                           # де вперше
            at[-1] / n_words,                          # де востаннє
            1.0 if at[0] == 0 else 0.0,                # чи це перше слово уривка
            math.log1p(len(candidate)),                # довжина рядка
            1.0 if candidate.isupper() else 0.0,       # абревіатура
            float(len(candidate.split())),             # скільки слів
            math.log1p(DF.get(low, 0) / max(N_DOCS, 1) * 1000),   # поширеність
            1.0 if kind == 'PROD' else 0.0,
            1.0 if kind == 'LIC' else 0.0,
            1.0 if kind == 'ORG' else 0.0,
            1.0 if kind == 'CAP' else 0.0,
        ]

        # --- QDEP: порівнюємо з питанням ---
        best_ctx = best_left = best_right = best_sent = best_near = 0.0
        for i in at:
            window = set(words[max(0, i - 10):i + 11])
            # скільки змістовних слів питання стоїть у вікні навколо кандидата;
            # ділимо на корінь із довжини питання, щоб довгі питання не вигравали
            best_ctx = max(best_ctx, len(window & q_content) / (len(q_content) ** 0.5 + 1))

            # точний збіг сусідів: скільки слів зліва від пропуску збігається
            # зі словами зліва від кандидата (і те саме справа)
            k = 0
            while k < len(q_left) and i - 1 - k >= 0 and words[i - 1 - k] == q_left[-1 - k]:
                k += 1
            best_left = max(best_left, float(k))
            k = 0
            while k < len(q_right) and i + 1 + k < len(words) and words[i + 1 + k] == q_right[k]:
                k += 1
            best_right = max(best_right, float(k))

            if q_positions:
                distance = min(abs(i - j) for j in q_positions)
                best_near = max(best_near, 1.0 / (1.0 + distance))

            around = set(words[max(0, i - 25):i + 26])
            best_sent = max(best_sent, len(around & q_content) / (len(around | q_content) + 1e-9))

        # чи ділить кандидат початок із якимось словом питання (TLS ↔ GnuTLS)
        prefix = 0.0
        for w in q_content:
            if len(w) >= 3 and len(low) >= 3 and (w.startswith(low) or low.startswith(w)):
                prefix = 1.0
                break

        row += [best_ctx, best_left, best_right,
                1.0 if low in q_all else 0.0,          # кандидат згаданий у питанні
                best_near, best_sent, prefix]
        rows.append(row)
    return item['cands'], rows


print('ознак на кандидата:', len(ALL_FEATURES))
print('з них не бачать питання:', len(PASS_FEATURES),
      '· бачать:', len(QDEP_FEATURES))

## 6 · Ознаки для трьох світів

Порахуємо ознаки тричі — на тих самих уривках, але з різними питаннями.

1. **Своє питання** — як має бути.
2. **Чуже питання** з випадкового **іншого** пакета. Тема інша, слова інші.
3. **Питання з того самого опису** — інше речення того самого пакета. Тема та
   сама, лексика та сама, але питають про **інше**.

Третій варіант — найсуворіший контроль із трьох. Якщо модель тримається на
чужому питанні з іншої теми, це ще можна списати на те, що тема допомагала.
Якщо вона тримається на сусідньому питанні того самого опису — вона не читає
питання взагалі, і жодного виправдання немає.

In [ ]:
mark = time.process_time()

FEAT_OWN = [features(q) for q in real]

# чуже питання з ІНШОГО пакета
order = list(range(len(real)))
random.Random(0).shuffle(order)
q_other = []
for i, j in enumerate(order):
    if real[j]['pkg'] == real[i]['pkg']:
        j = (j + 1) % len(real)
    q_other.append(real[j]['q'])
FEAT_OTHER = [features(q, text) for q, text in zip(real, q_other)]

# питання з ТОГО САМОГО опису
by_package = collections.defaultdict(list)
for i, q in enumerate(real):
    by_package[q['pkg']].append(i)
rng = random.Random(1)
FEAT_SIBLING = []
for i, q in enumerate(real):
    siblings = [j for j in by_package[q['pkg']] if real[j]['q'] != q['q']]
    FEAT_SIBLING.append(features(q, real[rng.choice(siblings)]['q']) if siblings else None)

n_siblings = sum(1 for f in FEAT_SIBLING if f is not None)
print('питань, у яких є «сусіднє» питання того самого опису:', n_siblings,
      f'({n_siblings/len(real):.1%})')
print('ознаки порахувано за', round(time.process_time() - mark, 1), 'с процесорного')

## 7 · Розбиття за пакетом — і чому не випадкове

Одна й та сама програма дає кілька питань, і всі вони мають **спільний уривок**.
Якби ми ділили питання випадково, частина питань про `gnutls` опинилась би в
навчальній частині, а частина — у перевірній, і модель бачила б той самий текст
з обох боків. Число вийшло б завищене й нічого не варте.

Тому ділимо **за пакетом**: усі питання однієї програми цілком ідуть або в
навчання, або в перевірку.

Розбиттів робимо **сорок**. Кожне розбиття — новий випадковий поділ пакетів, і
сорок чисел дають чесну картину розкиду. Пʼять розбиттів давали б надто вузьку
купу й спокусу написати «купи не перекриваються» там, де вони перекриваються.

І ще одне: **рубежі й модель міряємо на тих самих перевірних частинах**.
Порівнювати модель, перевірену на одній вибірці, з рубежем, порахованим на всьому
наборі, — це порівнювати різні речі.

In [ ]:
N_SPLITS = 40
TRAIN_SHARE = 0.7
LABELS = [[1 if c == q['ans'] else 0 for c in q['cands']] for q in real]


def fit_model(train_ids, columns, feature_bank):
    """Логістична регресія на кандидатах: 1 — правильна відповідь, 0 — решта."""
    X, y = [], []
    for i in train_ids:
        for row, label in zip(feature_bank[i][1], LABELS[i]):
            X.append([row[k] for k in columns])
            y.append(label)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(np.array(X), np.array(y))
    return model


def evaluate(model, test_ids, columns, feature_bank):
    """Точність: серед кандидатів беремо того, кому модель дала найбільший бал."""
    correct = total = 0
    for i in test_ids:
        if feature_bank[i] is None:
            continue
        candidates, rows = feature_bank[i]
        scores = model.decision_function(np.array([[r[k] for k in columns] for r in rows]))
        correct += candidates[int(np.argmax(scores))] == real[i]['ans']
        total += 1
    return correct / total


ALL_COLS = list(range(len(ALL_FEATURES)))
PASS_COLS = [INDEX_OF[name] for name in PASS_FEATURES]
package_list = sorted({q['pkg'] for q in real})
print('пакетів для розбиття:', len(package_list))
print('навчальних приблизно:', int(TRAIN_SHARE * len(package_list)))

In [ ]:
mark = time.process_time()
runs = collections.defaultdict(list)

for seed in range(N_SPLITS):
    shuffled = package_list[:]
    random.Random(seed).shuffle(shuffled)
    train_packages = set(shuffled[:int(TRAIN_SHARE * len(shuffled))])
    train_ids = [i for i, q in enumerate(real) if q['pkg'] in train_packages]
    test_ids = [i for i, q in enumerate(real) if q['pkg'] not in train_packages]
    test_items = [real[i] for i in test_ids]

    runs['випадковий кандидат'].append(pick_random_expected(test_items))
    runs['найчастіший кандидат'].append(accuracy(pick_most_frequent, test_items))

    blind = fit_model(train_ids, PASS_COLS, FEAT_OWN)
    runs['модель без питання'].append(evaluate(blind, test_ids, PASS_COLS, FEAT_OWN))

    full = fit_model(train_ids, ALL_COLS, FEAT_OWN)
    runs['модель повна'].append(evaluate(full, test_ids, ALL_COLS, FEAT_OWN))
    runs['контроль: чуже питання'].append(evaluate(full, test_ids, ALL_COLS, FEAT_OTHER))
    runs['контроль: питання того самого опису'].append(
        evaluate(full, test_ids, ALL_COLS, FEAT_SIBLING))

print('сорок розбиттів пораховано за', round(time.process_time() - mark, 1),
      'с процесорного')
print()
print(f'{"спосіб":38} {"середнє":>8}  купа по 40 розбиттях')
for name, values in runs.items():
    print(f'{name:38} {statistics.mean(values):8.4f}  '
          f'{min(values):.4f}..{max(values):.4f}')

## 8 · Купи перекриваються чи ні — і що робити, коли перекриваються

Дивитись на «найгірше…найкраще» двох способів і казати «купи не перекриваються»
можна лише тоді, коли вони справді не перекриваються. Але навіть коли
перекриваються, висновок ще не втрачено: **розбиття в нас спільні**, тож можна
порівнювати не купи, а **різницю на кожному розбитті окремо**.

Це набагато сильніший спосіб. Якщо модель обганяє рубіж на **всіх сорока**
розбиттях, то перекриття куп означає лише, що бувають легкі й важкі перевірні
вибірки, — а не що переваги немає.

In [ ]:
def paired(better, worse):
    """Різниця на кожному розбитті окремо. Розбиття спільні, тож це чесно."""
    diffs = [a - b for a, b in zip(runs[better], runs[worse])]
    wins = sum(1 for d in diffs if d > 0)
    overlap = not (min(runs[better]) > max(runs[worse]))
    print(f'{better}  ПРОТИ  {worse}')
    print(f'   середня різниця {statistics.mean(diffs):+.4f} '
          f'(від {min(diffs):+.4f} до {max(diffs):+.4f})')
    print(f'   виграно розбиттів: {wins} із {len(diffs)}')
    print(f'   купи {"ПЕРЕКРИВАЮТЬСЯ" if overlap else "не перекриваються"}')
    print()


paired('модель повна', 'найчастіший кандидат')
paired('модель повна', 'модель без питання')
paired('модель повна', 'контроль: чуже питання')
paired('модель повна', 'контроль: питання того самого опису')
paired('модель без питання', 'контроль: питання того самого опису')

## 8-а · Та сама модель на вужчому наборі

Числа вище отримані на наборі з великими літерами. А що буде, якщо взяти той
самий код і те саме навчання, але **вужчий розпізнавач сутностей** — той, що дав
653 питання й 51.91 % безкоштовних?

Це не риторичне питання. Якщо висновок про читання питання справді стосується
**задачі**, він має вистояти. Якщо він стосується лише **нашого набору** — це
теж треба знати, і краще дізнатись самому, ніж від рецензента.

In [ ]:
narrow = [q for q in fixed_plain if len(q['cands']) > 1]
narrow_pkgs = sorted({q['pkg'] for q in narrow})
print('вузький набір: змістовних питань', len(narrow), '· пакетів', len(narrow_pkgs))

# ознаки для вузького набору — своє питання і чуже
NARROW_OWN = [features(q) for q in narrow]
order_n = list(range(len(narrow)))
random.Random(0).shuffle(order_n)
q_other_n = []
for i, j in enumerate(order_n):
    if narrow[j]['pkg'] == narrow[i]['pkg']:
        j = (j + 1) % len(narrow)
    q_other_n.append(narrow[j]['q'])
NARROW_OTHER = [features(q, text) for q, text in zip(narrow, q_other_n)]
NARROW_Y = [[1 if c == q['ans'] else 0 for c in q['cands']] for q in narrow]


def fit_narrow(train_ids, columns, bank):
    X, y = [], []
    for i in train_ids:
        for row, label in zip(bank[i][1], NARROW_Y[i]):
            X.append([row[k] for k in columns])
            y.append(label)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(np.array(X), np.array(y))
    return model


def eval_narrow(model, test_ids, columns, bank):
    correct = 0
    for i in test_ids:
        candidates, rows = bank[i]
        scores = model.decision_function(np.array([[r[k] for k in columns] for r in rows]))
        correct += candidates[int(np.argmax(scores))] == narrow[i]['ans']
    return correct / len(test_ids)


narrow_runs = collections.defaultdict(list)
for seed in range(N_SPLITS):
    shuffled = narrow_pkgs[:]
    random.Random(seed).shuffle(shuffled)
    train_packages = set(shuffled[:int(TRAIN_SHARE * len(shuffled))])
    train_ids = [i for i, q in enumerate(narrow) if q['pkg'] in train_packages]
    test_ids = [i for i, q in enumerate(narrow) if q['pkg'] not in train_packages]
    if not train_ids or not test_ids:
        continue
    test_items = [narrow[i] for i in test_ids]
    narrow_runs['випадковий кандидат'].append(pick_random_expected(test_items))
    narrow_runs['найчастіший кандидат'].append(accuracy(pick_most_frequent, test_items))
    blind = fit_narrow(train_ids, PASS_COLS, NARROW_OWN)
    narrow_runs['модель без питання'].append(eval_narrow(blind, test_ids, PASS_COLS, NARROW_OWN))
    full = fit_narrow(train_ids, ALL_COLS, NARROW_OWN)
    narrow_runs['модель повна'].append(eval_narrow(full, test_ids, ALL_COLS, NARROW_OWN))
    narrow_runs['контроль: чуже питання'].append(
        eval_narrow(full, test_ids, ALL_COLS, NARROW_OTHER))

print()
print(f'{"спосіб":28} {"великий набір":>14} {"вузький набір":>14}')
for name in ('випадковий кандидат', 'найчастіший кандидат', 'модель без питання',
             'модель повна', 'контроль: чуже питання'):
    print(f'{name:28} {statistics.mean(runs[name]):14.4f} '
          f'{statistics.mean(narrow_runs[name]):14.4f}')
print()
print('купа «найчастіший кандидат» на вузькому: '
      f'{min(narrow_runs["найчастіший кандидат"]):.4f}..'
      f'{max(narrow_runs["найчастіший кандидат"]):.4f}')
print('купа «модель повна» на вузькому:         '
      f'{min(narrow_runs["модель повна"]):.4f}..'
      f'{max(narrow_runs["модель повна"]):.4f}')
diffs = [a - b for a, b in zip(narrow_runs['модель повна'],
                               narrow_runs['найчастіший кандидат'])]
print()
print(f'парна різниця «повна − найчастіший» на вузькому: {statistics.mean(diffs):+.4f} '
      f'({min(diffs):+.4f}..{max(diffs):+.4f})')
print(f'виграно розбиттів: {sum(1 for d in diffs if d > 0)} із {len(diffs)}')
print(f'падіння від підміни питання: '
      f'{statistics.mean(narrow_runs["контроль: чуже питання"]) - statistics.mean(narrow_runs["модель повна"]):+.4f}')

Ось і відповідь, і вона незручна.

На вузькому наборі та сама модель, навчена тим самим кодом, **програє тупому
рубежу**, а контроль із підміненим питанням забирає мізер і не доводить нічого.
Купи при цьому величезні — ширші за всю різницю, яку ми обговорюємо, бо пакетів
там усього сотня.

Отже висновок «тест міряє читання» — це твердження **про наш великий набір**, а
не про cloze-конструкцію взагалі. На тій самій конструкції з іншим розпізнавачем
сутностей його зробити було б неможливо: система нічого не вміє, а система, яка
нічого не вміє, не доводить нічого й про дані.

Тепер можна вимовити висновок, і він мусить спиратись рівно на ці числа.

Порівняй три речі: наскільки повна модель обганяє **найчастішого кандидата**,
наскільки вона обганяє **саму себе без питання**, і наскільки вона **падає**,
коли питання підмінили. Якщо падіння з підміненим питанням доходить до рівня
моделі, що питання не бачить, — модель справді читала питання, і набір справді
міряє читання.

Якщо ж підміна питання нічого не зіпсувала — усе, що ти виміряв, було
частотністю кандидатів, а не розумінням.

In [ ]:
full_mean = statistics.mean(runs['модель повна'])
blind_mean = statistics.mean(runs['модель без питання'])
freq_mean = statistics.mean(runs['найчастіший кандидат'])
other_mean = statistics.mean(runs['контроль: чуже питання'])
sibling_mean = statistics.mean(runs['контроль: питання того самого опису'])

print(f'рубіж «найчастіший кандидат»           {freq_mean:.4f}')
print(f'модель, що питання не бачить           {blind_mean:.4f}')
print(f'модель повна                           {full_mean:.4f}')
print(f'вона ж із чужим питанням               {other_mean:.4f}')
print(f'вона ж із питанням того самого опису   {sibling_mean:.4f}')
print()
print(f'читання питання додає                  {full_mean - blind_mean:+.4f}')
print(f'підміна питання забирає                {sibling_mean - full_mean:+.4f}')
print()
if full_mean > freq_mean and sibling_mean < blind_mean:
    print('Модель обганяє тупий рубіж І падає нижче своєї ж сліпої версії,')
    print('коли питання підмінили. Отже набір міряє саме читання питання.')
elif full_mean > freq_mean:
    print('Модель обганяє тупий рубіж, але підміна питання її не зламала.')
    print('Виграш дала не робота з питанням — шукай, яка ознака його дала.')
else:
    print('Модель не обганяє тупого рубежу. Про сам набір це не каже нічого:')
    print('слабка система не є доводом про задачу.')

## 9 · Які ознаки виявились важливими

Навчимо модель на всьому наборі й подивимось на ваги. Ознаки стандартизовані —
кожна зведена до нульового середнього й одиничного розкиду, — тому ваги можна
порівнювати між собою.

Знак важить не менше за величину. Наприклад, ознака «кандидат уже згаданий у
питанні» майже напевно вийде з **мінусом**: те, що вже назвали в питанні, рідше
є відповіддю.

In [ ]:
final_model = fit_model(list(range(len(real))), ALL_COLS, FEAT_OWN)
weights = final_model[-1].coef_[0]

print(f'{"ознака":16} {"вага":>8}   група')
for name, value in sorted(zip(ALL_FEATURES, weights), key=lambda t: -abs(t[1])):
    group = 'читає питання' if name in QDEP_FEATURES else '—'
    print(f'{name:16} {value:+8.3f}   {group}')

### Перевірка: усередині бібліотеки немає магії

`decision_function` у логістичної регресії — це просто скалярний добуток
стандартизованих ознак на ваги плюс вільний член. Порахуємо його руками й
переконаємось, що збігається.

In [ ]:
scaler = final_model[0]
regression = final_model[-1]

sample_candidates, sample_rows = FEAT_OWN[0]
X = np.array(sample_rows)

# те саме, що робить бібліотека: (x − середнє) / розкид, потім x·w + b
standardized = (X - scaler.mean_) / scaler.scale_
by_hand = standardized @ regression.coef_[0] + regression.intercept_[0]
by_library = final_model.decision_function(X)

assert np.allclose(by_hand, by_library), 'розрахунок розійшовся!'
print('✅ збігається')
print('бали кандидатів першого питання:')
for candidate, score in zip(sample_candidates, by_hand):
    mark_ = ' ← правильна' if candidate == real[0]['ans'] else ''
    print(f'   {candidate:<24} {score:+.4f}{mark_}')

## 10 · Питання, на які відповіді немає

Досі кожне питання мало відповідь в уривку — ми самі це вимагали, коли будували
набір. У житті так не буває: людина ставить питання, а в тексті відповіді немає.

Такі питання в нас уже є, і задарма: щоразу, коли викинута сутність **не**
траплялась в уривку, ми питання відкидали. Зберемо їх назад і подивимось, що
робить із ними наша модель.

Робить вона рівно одне: **відповідає**. Вона не вміє інакше — її будували так,
щоб обрати найкращого з кандидатів, і найкращий є завжди.

In [ ]:
def make_unanswerable(packages):
    """Ті самі cloze-питання, але відповіді в уривку НЕМАЄ."""
    items = []
    for p in packages:
        parts = sentences(p['desc'])
        if len(parts) < 2:
            continue
        for i, sentence in enumerate(parts):
            passage = ' '.join(parts[:i] + parts[i + 1:])
            occurrences = find_entities(passage)
            candidates = sorted({o[2] for o in occurrences})
            if len(candidates) < 2:
                continue
            for start, end, string, kind in find_entities(sentence):
                if string in candidates:
                    continue                  # це якраз питання З відповіддю
                items.append({'pkg': p['name'], 'src': 'опис', 'passage': passage,
                              'occ': occurrences,
                              'q': sentence[:start] + '@blank' + sentence[end:],
                              'ans': string, 'type': kind, 'cands': candidates})
    return items


unanswerable = make_unanswerable(packages)
FEAT_NONE = [features(q) for q in unanswerable]
share = len(unanswerable) / (len(unanswerable) + len(real))
print('питань без відповіді в уривку:', len(unanswerable))
print('питань з відповіддю          :', len(real))
print('частка нерозвʼязних          :', round(share, 4))

Тепер дамо моделі обидва набори разом і порахуємо **спільну точність**:
питання з відповіддю зараховується, коли модель назвала правильну сутність;
питання без відповіді — коли модель сказала «не знаю».

Спершу без права мовчати. Модель відповідає завжди, тож усі нерозвʼязні питання —
помилки за визначенням.

Потім дамо їй **поріг**: якщо найкращий бал нижчий за поріг, вона каже «не знаю».
Поріг доберемо на навчальній частині, а оголосимо на перевірній — інакше це буде
підгонка.

In [ ]:
mark = time.process_time()
without_abstain, with_abstain, best_thresholds = [], [], []

for seed in range(5):                       # порогові прогони дорожчі, тут досить пʼятьох
    shuffled = package_list[:]
    random.Random(seed).shuffle(shuffled)
    train_packages = set(shuffled[:int(TRAIN_SHARE * len(shuffled))])
    train_ids = [i for i, q in enumerate(real) if q['pkg'] in train_packages]
    test_ids = [i for i, q in enumerate(real) if q['pkg'] not in train_packages]
    model = fit_model(train_ids, ALL_COLS, FEAT_OWN)

    def best_score_and_pick(bank, i):
        candidates, rows = bank[i]
        scores = model.decision_function(np.array(rows))
        k = int(np.argmax(scores))
        return float(scores[k]), candidates[k]

    # перевірна частина: питання з відповіддю й питання без неї
    yes = [best_score_and_pick(FEAT_OWN, i) + (real[i]['ans'],) for i in test_ids]
    no_ids = [i for i, q in enumerate(unanswerable) if q['pkg'] not in train_packages]
    no = [best_score_and_pick(FEAT_NONE, i)[0] for i in no_ids]
    total = len(yes) + len(no)

    without_abstain.append(sum(1 for s, pick, gold in yes if pick == gold) / total)

    # поріг добираємо на НАВЧАЛЬНІЙ частині, щоб не підганяти під перевірну
    tr_yes = [best_score_and_pick(FEAT_OWN, i) + (real[i]['ans'],) for i in train_ids]
    tr_no_ids = [i for i, q in enumerate(unanswerable) if q['pkg'] in train_packages]
    tr_no = [best_score_and_pick(FEAT_NONE, i)[0] for i in tr_no_ids]
    grid = np.quantile(np.array([s for s, _, _ in tr_yes] + tr_no), np.linspace(0.02, 0.98, 49))
    scored = [(sum(1 for s, pick, gold in tr_yes if s >= t and pick == gold)
               + sum(1 for s in tr_no if s < t), t) for t in grid]
    threshold = max(scored)[1]
    best_thresholds.append(float(threshold))

    with_abstain.append(
        (sum(1 for s, pick, gold in yes if s >= threshold and pick == gold)
         + sum(1 for s in no if s < threshold)) / total)

print('спільна точність без права мовчати :',
      round(statistics.mean(without_abstain), 4),
      f'({min(without_abstain):.4f}..{max(without_abstain):.4f})')
print('спільна точність із порогом        :',
      round(statistics.mean(with_abstain), 4),
      f'({min(with_abstain):.4f}..{max(with_abstain):.4f})')
print('поріг, дібраний на навчальній      :',
      round(statistics.mean(best_thresholds), 3))
print('пораховано за', round(time.process_time() - mark, 1), 'с процесорного')

## 11 · Дві метрики: точний збіг і F1 по токенах

Дотепер ми рахували **точний збіг** (exact match, EM): відповідь або та сама, або
ні. Це сувора метрика, і вона карає за дрібницю однаково з грубою помилкою.

Тому поруч завжди звітують **F1 по токенах**. Відповідь і еталон ріжуть на слова
й дивляться, скільки слів спільних:

- **точність** = скільки слів відповіді є в еталоні, поділити на довжину відповіді;
- **повнота** = скільки слів еталона є у відповіді, поділити на довжину еталона;
- **F1** = їхнє гармонійне середнє, тобто 2 × точність × повнота ÷ (точність + повнота).

Порахуємо руками на прикладі, а потім звіримо з нашою функцією.

In [ ]:
def token_f1(prediction, gold):
    """F1 по токенах між відповіддю моделі й еталоном."""
    p = [w.lower() for w in tokens(prediction)]
    g = [w.lower() for w in tokens(gold)]
    shared = sum((collections.Counter(p) & collections.Counter(g)).values())
    if shared == 0:
        return 0.0
    precision = shared / len(p)
    recall = shared / len(g)
    return 2 * precision * recall / (precision + recall)


# рахуємо руками: відповідь «Perl compatible library», еталон «Perl compatible»
# спільних слів 2; точність 2/3 = 0.6667; повнота 2/2 = 1.0
# F1 = 2 × 0.6667 × 1.0 / (0.6667 + 1.0) = 0.8
by_hand = 2 * (2 / 3) * 1.0 / ((2 / 3) + 1.0)
print('руками   :', round(by_hand, 4))
print('функцією :', round(token_f1('Perl compatible library', 'Perl compatible'), 4))
assert abs(by_hand - token_f1('Perl compatible library', 'Perl compatible')) < 1e-12
print('✅ збігається')
print()
for prediction, gold in (('TLS', 'TLS'), ('SSL', 'TLS'),
                         ('PCRE2', 'PCRE'), ('GNU General Public License', 'General Public')):
    print(f'{prediction:<26} проти {gold:<16} EM={int(prediction == gold)}  '
          f'F1={token_f1(prediction, gold):.4f}')

Видно головне: `PCRE2` проти `PCRE` дає F1 **нуль**, бо це різні токени, — а
`GNU General Public License` проти `General Public` дає непоганий F1 при нульовому
точному збігу. Обидві метрики потрібні саме тому, що жодна не покриває іншу.

Порахуємо обидві на нашій моделі.

In [ ]:
mark = time.process_time()
exact_runs, f1_runs = [], []
for seed in range(5):
    shuffled = package_list[:]
    random.Random(seed).shuffle(shuffled)
    train_packages = set(shuffled[:int(TRAIN_SHARE * len(shuffled))])
    train_ids = [i for i, q in enumerate(real) if q['pkg'] in train_packages]
    test_ids = [i for i, q in enumerate(real) if q['pkg'] not in train_packages]
    model = fit_model(train_ids, ALL_COLS, FEAT_OWN)

    exact = f1_sum = 0.0
    for i in test_ids:
        candidates, rows = FEAT_OWN[i]
        pick = candidates[int(np.argmax(model.decision_function(np.array(rows))))]
        exact += pick == real[i]['ans']
        f1_sum += token_f1(pick, real[i]['ans'])
    exact_runs.append(exact / len(test_ids))
    f1_runs.append(f1_sum / len(test_ids))

print('точний збіг (EM):', round(statistics.mean(exact_runs), 4),
      f'({min(exact_runs):.4f}..{max(exact_runs):.4f})')
print('F1 по токенах   :', round(statistics.mean(f1_runs), 4),
      f'({min(f1_runs):.4f}..{max(f1_runs):.4f})')
print('різниця         :', round(statistics.mean(f1_runs) - statistics.mean(exact_runs), 4))
print('пораховано за', round(time.process_time() - mark, 1), 'с процесорного')

## 12 · Межі того, що ми тут виміряли

Скажімо це прямо, бо жодна з обставин нижче з таблиць не видна.

1. **Питання ставила не людина, а конструкція.** Cloze-питання завжди має
   відповідь у тексті, завжди одну й завжди у формі сутності. Справжні питання
   бувають без відповіді, з кількома відповідями і з відповіддю, яку треба
   зібрати з двох речень.
2. **Розмітка срібна.** Сутності знайдено газетиром і великою літерою, а не
   людиною. Частина «сутностей» — звичайні слова на початку речення.
3. **Тексти однорідні.** Описи пакетів написані в одному жанрі, і модель могла
   вивчити жанр, а не мову.
4. **Набір пакетів у тебе інший**, тому всі числа тут — твої, а не наші.
5. **Модель лінійна й на ознаках.** Це навмисно: її видно наскрізь. Нейромережа
   дала б більше, але приховала б, звідки береться кожен бал.

І головне, що варто винести з цього зошита як звичку: **будь-який результат у
QA треба перевіряти контролем «а що буде, якщо не читати питання»**. Він коштує
десять рядків коду й рятує від висновку, який не має під собою нічого.

In [ ]:
print('процесорного часу на весь зошит:', round(time.process_time() - START, 1), 'с')

## Завдання

**🟢 Рівень 1.** Постав `MIN_WORDS = 12` замість 20 і перебудуй набір. Скільки
стало питань, скільки безкоштовних і як змінився рубіж «найчастіший кандидат»?
Поясни напрямок зміни.

**🟡 Рівень 2.** Прибери з `QDEP_FEATURES` ознаку `in_question` і перенавчи
модель на тих самих сорока розбиттях. На скільки впала точність? Зроби те саме
для `near_qword`. Яка з ознак важливіша — і чи збігається це з їхніми вагами?

**🔴 Рівень 3.** Побудуй третій контроль: лиши питання своїм, але **перемішай
слова в ньому** випадково. Порядок слів зникає, лексика лишається. Якщо
точність не впаде — модель бачить у питанні мішок слів, а не речення. Порівняй
падіння з тим, що дала підміна питання.

Повний опис — у [homework.html](homework.html).